# **0. Matériel : vérifier que le GPU est bien sélectionné (A COMPLETER)**

In [ ]:
!nvidia-smi

# **1. Connexion à la plateforme Wandb** ##

Weights and Biases (W&B) est une plateforme qui permet de suivre, visualiser et partager les résultats d'expériences en temps réel. Cela permet de faciliter le tracking des expériences. Veuillez Créer vous un compte en allant sur le lien [https://wandb.ai/site](https://wandb.ai/site)

In [ ]:
!pip install wandb
!wandb login

# **2.1. Importation des libraries** ##



In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, SubsetRandomSampler
import matplotlib.pyplot as plt
import cv2
import torchvision.datasets as datasets
import torchvision.models as models
import torchvision.transforms.v2 as transforms
from tqdm import tqdm
import wandb
!pip install pytorch_lightning
from torchmetrics import Accuracy
from torchmetrics.classification import MulticlassConfusionMatrix, Accuracy
import pytorch_lightning as pl
from pytorch_lightning.loggers import WandbLogger, TensorBoardLogger , CSVLogger
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from PIL import Image

# **2.2 Installation de libraririe "benchamrk" pour évaluer les résultats en détail** ##

In [ ]:
!pip install pytorch_bench
from pytorch_bench import benchmark

Benchmark : librarire développée au sein de l'équipe Deep'ILIA (Service ILIA-FPMS) dans le cadre de la thèse de doctorat de Maxime Gloesener.

# **3.1. Téléchargement du jeu de données d'entraînement** ##


In [ ]:
!rm -rf sample_data
!wget https://nextcloud.ig.umons.ac.be/s/QP7FoyttxJgS8Cj/download/small.zip
!mkdir -p FIRE/train
!unzip small.zip -d FIRE/train
!rm small.zip

In [ ]:
# DB1 : https://nextcloud.ig.umons.ac.be/s/REWbK6K4XRtoeNw/download/FIRE_DATABASE_1.zip
!wget https://nextcloud.ig.umons.ac.be/s/REWbK6K4XRtoeNw/download/FIRE_DATABASE_1.zip
!unzip FIRE_DATABASE_1.zip -d FIRE/train
!rm FIRE_DATABASE_1.zip
!mv FIRE/train/FIRE_DATABASE_1 FIRE/train/DB1
# DB2 : https://nextcloud.ig.umons.ac.be/s/faKyDy7LCxfz9Xk/download/FIRE_DATABASE_2.zip
!wget https://nextcloud.ig.umons.ac.be/s/faKyDy7LCxfz9Xk/download/FIRE_DATABASE_2.zip
!unzip FIRE_DATABASE_2.zip -d FIRE/train
!rm FIRE_DATABASE_2.zip
!mv FIRE/train/FIRE_DATABASE_2 FIRE/train/DB2
# DB3 : https://nextcloud.ig.umons.ac.be/s/KaqzczZsXfsnMER/download/FIRE_DATABASE_3.zip
!wget  https://nextcloud.ig.umons.ac.be/s/KaqzczZsXfsnMER/download/FIRE_DATABASE_3.zip
!unzip FIRE_DATABASE_3.zip -d FIRE/train
!rm FIRE_DATABASE_3.zip
!mv FIRE/train/FIRE_DATABASE_3 FIRE/train/DB3




# **3.2. Téléchargement du jeu de données de test** ##

In [ ]:
!wget https://nextcloud.ig.umons.ac.be/s/XqEMQtqQNPoG2cY/download/test.zip
!unzip test.zip -d FIRE/test
!rm test.zip

#**4. Paramètres d'entrainement "Hyper-paramètres"**

In [ ]:
Train_data_path = "DB3" #@param ["small","DB1","DB2","DB3"]
Train_data_path = os.path.join('FIRE/train', Train_data_path)
Test_data_path = "test" #@param ["test","test_defi1"]
Test_data_path = os.path.join('FIRE/test', Test_data_path)
Batch_size=64 #@param [8,16,32,64,128,256] {type:"raw"}
Epochs=50 #@param [1,5, 10,15,20,30,50,100,200] {type:"raw"}
Learning_rate = 0.001 #@param [0.1, 0.01,0.02,0.05,0.001,0.002,0.005] {type:"raw"}
Train_split = 0.8 #@param [0.7,0.8,0.9] {type:"raw"}
Img_size = 224 #@param [224,299] {type:"raw"}
Accelerator= "auto" #@param ["cpu","gpu","auto"]
num_classes = 3
LOG_DIR="logs/"

#**5.1.Télécharger modèle préentrainé VGG16 & afficher l'architecture (A COMPLETER)**

In [ ]:
# Détecter si un GPU est disponible pour rassurer
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Charger le modèle VGG16 pré-entraîné
res_mod = models.mobilenet_v2(pretrained=True)

# Transférer le modèle sur le GPU si disponible
res_mod.to(device)

# Afficher la structure du modèle
print(res_mod)

# **5.2. Définir modèle, param Forward/Backward & Transfer Leraning (A COMPLETER)** ##

In [ ]:
class FireDetectionModel(pl.LightningModule):
    def __init__(self, num_classes=num_classes, learning_rate=Learning_rate,dropout_rate=0.5):
        super().__init__()
        self.mobilenet_v2 = models.mobilenet_v2(pretrained=True)        # Récupérer un modèle pré-entrainé : par exemple VGG16
        num_ftrs = self.mobilenet_v2.classifier[-1].in_features  # Get the number of features from the penultimate layer
        self.mobilenet_v2.classifier[-1] = nn.Sequential(
             nn.Dropout(p=dropout_rate),
             nn.Linear(num_ftrs, num_classes)
           )  # Dernière couche entièrement connectée

        self.criterion = nn.CrossEntropyLoss()
        self.learning_rate = learning_rate
        self.num_classes = num_classes
        self.confusion_matrix = MulticlassConfusionMatrix(num_classes=self.num_classes)
        self.test_accuracy = Accuracy(task="multiclass", num_classes=self.num_classes)
        self.val_accuracy = Accuracy(task="multiclass", num_classes=self.num_classes)
        self.train_accuracy = Accuracy(task="multiclass", num_classes=self.num_classes)


    def forward(self, x):
        features = self.mobilenet_v2(x)  #Passer les données en entrée à travers le modèle
        return features   #Renvoie les sorties aprés passage dans le modéle

    def training_step(self, batch, batch_idx):
        #il extraits les inputs(images) et les labels du batch fournie par le dataloader
        inputs, labels = batch
        outputs = self(inputs)
        loss = self.criterion(outputs, labels)
        acc = self.train_accuracy(outputs, labels)
        #c'est pour enregistrer et visualiser la perte
        self.log_dict({'train_loss':loss,"train_acc":acc}, on_step=True,prog_bar=True,logger=True, on_epoch=True)
        return loss

    def on_train_epoch_end(self):
        self.train_accuracy.reset()

    def validation_step(self, batch, batch_idx):
        inputs, labels = batch
        outputs = self(inputs)
        val_loss = self.criterion(outputs, labels)
        val_acc=self.val_accuracy(outputs, labels)
        self.log_dict({'val_loss':val_loss,"val_acc":val_acc},prog_bar=True, on_step=False, on_epoch=True)
        return  val_loss


    def on_validation_epoch_end(self):
        self.val_accuracy.reset()

    def test_step(self, batch, batch_idx):
        inputs, labels = batch
        outputs = self(inputs)
        test_loss = self.criterion(outputs, labels)
        test_acc = self.test_accuracy(outputs, labels)
        self.log_dict({'test_loss':test_loss,"test_acc":test_acc},prog_bar=True, on_step=False, on_epoch=True)
        self.confusion_matrix.update(outputs.argmax(dim=1), labels)

        return test_loss


    def on_test_end(self):
        self.test_accuracy.reset()
        # Compute confusion matrix at the end of the test epoch
        fig_, ax_ = self.confusion_matrix.plot()
        # Reset the confusion matrix
        self.confusion_matrix.reset()

    def configure_optimizers(self):
        return torch.optim.SGD(self.parameters(), lr=self.learning_rate)




# **6. Afficher quelques images avec la fonction "display_class_images"(A COMPLETER)** ##

In [ ]:
def display_class_images(class_path):
  import glob
  import matplotlib.image as mpimg
  images = []
  for img_path in glob.glob(class_path):
      images.append(mpimg.imread(img_path))
  plt.figure(figsize=(14,12))
  columns = 4
  for i, img in enumerate(images):
      if (i<=20):
        img=cv2.resize(img, (256,256))
        plt.subplot(5, 5, i + 1)
        plt.imshow(img)

In [ ]:
class_path = '/content/FIRE/train/small/fire/*.jpg'
display_class_images(class_path)


In [ ]:
class_path = '/content/FIRE/train/small/no_fire/*.jpg'
display_class_images(class_path)

In [ ]:
class_path = '/content/FIRE/train/small/start_fire/*.jpg'
display_class_images(class_path)

# **7. Création des jeux de données d'entraînement, validation et test "Data Loaders"** #

In [ ]:
def create_data_loaders(dataset_path, testset_path, batch_size, train_split, img_size):
    transform = transforms.Compose([
        transforms.RandomResizedCrop(img_size),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
    ])

    dataset = datasets.ImageFolder(dataset_path, transform=transform)
    dataset_size = len(dataset)
    indices = list(range(dataset_size))
    split = int(np.floor(train_split * dataset_size))
    np.random.shuffle(indices)
    train_indices, val_indices = indices[:split], indices[split:]

    train_sampler = SubsetRandomSampler(train_indices)
    val_sampler = SubsetRandomSampler(val_indices)

    train_loader = DataLoader(dataset, batch_size=batch_size, sampler=train_sampler)
    val_loader = DataLoader(dataset, batch_size=batch_size, sampler=val_sampler)

    test_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
    ])
    test_dataset = datasets.ImageFolder(testset_path, transform=test_transform)
    test_loader = DataLoader(test_dataset, batch_size=batch_size)

    return train_loader, val_loader, test_loader

# **8. Définir les hyper-paramètres, EarlyStopping, Checkpoints et paramètres WandDB"** #

In [ ]:
# Initialize wandb
wandb.init(project="fire-detection", config={
    "learning_rate": Learning_rate,
    "epochs": Epochs,
    "batch_size": Batch_size,
    "model": "mobilenet_V2"
})

# Create data loaders
train_loader, val_loader, test_loader = create_data_loaders(
    Train_data_path, Test_data_path, Batch_size, Train_split, Img_size
)

# Initialize model
model = FireDetectionModel(num_classes=3, learning_rate=Learning_rate)
#x = torch.randn(1, 3, 224, 224)  # Exemple de tenseur d'entrée
#y = model(x)

# Générer et enregistrer le graphique
#make_dot(y, params=dict(model.named_parameters())).render("fire_detection_model", format="png")
# Setup callbacks
checkpoint_callback = ModelCheckpoint(
    monitor='val_loss',
    dirpath='checkpoints',
    filename='best-checkpoint',
    save_top_k=1,
    mode='min'
)
early_stop_callback = EarlyStopping(
    monitor='val_loss',
    patience=10,
    mode='min'
)


# Initialize WandbLogger
wandb_logger = WandbLogger(project="fire-detection")

# Initialize WandbLogger
csv_logger = CSVLogger(LOG_DIR, name="cnn", version='')

# Initialize Trainer
trainer = pl.Trainer(
    max_epochs=Epochs,
    accelerator=Accelerator,
    log_every_n_steps=1,
    devices=1,
    logger=[wandb_logger, csv_logger],
    callbacks=[checkpoint_callback, early_stop_callback],
)

# **9. Lancer l'entrainement (A COMPLETER)** #

In [ ]:
trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)

# **10. Evaluer le modèle (A COMPLETER)** ##

In [ ]:
# Test the model
test_results = trainer.test(model, dataloaders=test_loader)

# **10.1 Exporter le modèle en .jit**

In [ ]:
jit_model = model.to_torchscript()
torch.jit.save(jit_model, 'model_jit.pth')
from google.colab import files
files.download('model_jit.pth')

# **11. Afficher les courbes d'entrainement avec la fonction "plot_metrics"** ##

In [ ]:
def plot_metrics(log_folder):
  import pandas as pd
  import matplotlib.pyplot as plt

  # Load the CSV file generated by CSVLogger
  df = pd.read_csv(f'{LOG_DIR}/{log_folder}/metrics.csv')
  train_df = df[df['train_loss_epoch'].notna()]
  val_df = df[df['val_loss'].notna()]

  # Plot training loss
  plt.plot(train_df['epoch'], train_df['train_loss_epoch'], label='Train Loss')
  plt.plot(val_df['epoch'], val_df['val_loss'], label='Validation Loss')
  plt.xlabel('Epoch')
  plt.ylabel('Loss')
  plt.title('Training & Validation Loss')
  plt.legend()
  plt.grid(True)
  plt.show()

  # Plot training accuracy
  plt.plot(train_df['epoch'], train_df['train_acc_epoch'], label='Train Acc')
  plt.plot(val_df['epoch'], val_df['val_acc'], label='Val Acc')
  plt.xlabel('Epoch')
  plt.ylabel('Accuracy')
  plt.title('Training & Validation Accuracy')
  plt.legend()
  plt.grid(True)
  plt.show()

In [ ]:
plot_metrics(log_folder="cnn")

# **12. Evaluation du modèle selon différentes métriques de notre librarie "Benchmark"** #

In [ ]:
example_input = torch.randn(1, 3, Img_size, Img_size)
results = benchmark(model, example_input)

# log to wandb
wandb.log({
    "benchmark": results
})
wandb.finish()


# **13. Tester le modèle avec une image de test de votre choix (A COMPLETER)"**

In [ ]:
image_path = 'img1.jpg'
classes = ["fire", "no fire", "start fire"]

# Load and preprocess the image
img = Image.open(image_path).convert('RGB')
transform = transforms.Compose([
    transforms.Resize((Img_size, Img_size)),
    transforms.ToTensor(),
])
x = transform(img).unsqueeze(0).to("cuda")  # Add batch dimension
model.to("cuda")
# Predict
model.eval()
with torch.no_grad():
    pred = model(x)
    probabilities = torch.nn.functional.softmax(pred[0], dim=0)

# Display results
plt.figure(figsize=(10, 10))
img = cv2.imread(image_path)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

for pos, prob in enumerate(probabilities):
    class_name = classes[pos]
    print(f"Class Name: {class_name} --- Class Probability: {prob.item()*100:.2f}%")

    if pos == torch.argmax(probabilities).item():
        font = cv2.FONT_HERSHEY_COMPLEX
        textsize = cv2.getTextSize(class_name, font, 1, 2)[0]
        textX = (img.shape[1] - textsize[0]) // 2
        textY = (img.shape[0] + textsize[1]) // 2
        cv2.putText(img, class_name, (textX-10, textY), font, 2, (255,0,0), 6, cv2.LINE_AA)

plt.imshow(img)
plt.axis('off')
plt.show()

Traitent  en temps réel

In [ ]:
import cv2
import numpy as np
from skimage.metrics import structural_similarity as ssim
from google.colab.patches import cv2_imshow  # Importer cv2_imshow de Google Colab
import torch
from torchvision import transforms
from PIL import Image

# Liste des classes
classes = ["fire", "no fire", "start fire"]

# Charger le modèle TorchScript
model = torch.jit.load('model_jit.pth')
model.eval()

# Transformer pour l'entrée du modèle
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# Charger la vidéo
cap = cv2.VideoCapture('video.mp4')
if not cap.isOpened():
    print("Erreur lors de l'ouverture de la vidéo.")
    exit()

# Récupère le nombre de frames par seconde de la vidéo
fps = cap.get(cv2.CAP_PROP_FPS)

# Frame précédente et score SSIM initial
prev_frame = None  #Variable qui stockera la précédente frame de la vidéo pour la comparaison.
prev_score = 0    # Variable qui stocke le score SSIM entre la frame actuelle et la précédente.

while True:
    ret, frame = cap.read()
    if not ret:
        break  # Si la vidéo est terminée, sortir de la boucle

    # Convertir la frame actuelle en niveaux de gris pour le calcul du SSIM
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Calculer le SSIM si une frame précédente existe
    score = 0  # Initialiser le score SSIM
    if prev_frame is not None:
        score = ssim(prev_frame, gray)

        # Si la similarité est trop élevée, passer à la frame suivante
        if score > 0.95:  # Ajuster le seuil de similarité si nécessaire
            continue

    # Mettre à jour la frame précédente
    prev_frame = gray

    # Prétraitement de la frame pour le modèle
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    frame_pil = Image.fromarray(frame_rgb)
    input_tensor = transform(frame_pil).unsqueeze(0)

    # Prédiction avec le modèle
    with torch.no_grad():
        output = model(input_tensor)
        predicted_class = torch.argmax(output, dim=1).item()

    # Ajouter la classe prédite sur la frame
    label = f'Classe : {classes[predicted_class]}'
    cv2.putText(frame, label, (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    # Afficher la frame dans Colab
    cv2_imshow(frame)

    # Attendre pour maintenir l'intervalle entre les frames, en fonction des FPS
    key = cv2.waitKey(int(1000 / fps))  # Ajuster le délai avec les FPS
    if key == 27:  # Sortir si la touche 'Esc' est pressée
        break

cap.release()
cv2.destroyAllWindows()